In [1]:
import pandas as pd
import nfl_data_py as nfl
import numpy as np
import plotly.express as px
import plotly.figure_factory as ffact
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.io as pio
from IPython.display import Image

import football_functions_public as ff

In [2]:
year = range(2018,2025)
df = nfl.import_weekly_data(year,['player_id','player_display_name', 'position',
       'headshot_url', 'recent_team', 'season', 'season_type','week',
      'completions', 'attempts', 'passing_yards',
       'passing_tds', 'interceptions', 'passing_air_yards', 'passing_yards_after_catch',
      'passing_epa', 'pacr', 
      'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_air_yards', 'receiving_yards_after_catch',
      'receiving_epa', 'racr', 'target_share', 'air_yards_share', 'wopr', 
       'fantasy_points_ppr'], False)
df = df.query("season_type == 'REG'")

ngs_pass_df = nfl.import_ngs_data('passing', year)
ngs_rush_df = nfl.import_ngs_data('rushing', year)
ngs_rec_df = nfl.import_ngs_data('receiving', year)

df.rename(columns={"player_display_name": "name"}, inplace = True)
df['name'] = ff.fixname(df['name'])
ID_dict = df[['name', 'player_id']].drop_duplicates().set_index('name').to_dict()['player_id']

directory = 'D:/OneDrive/Documents/Python Scripts/football/PFF'
QBs_Summary = ff.QB_season_recap(year,ngs_pass_df,df, ID_dict, directory)
QBs_Summary = QBs_Summary.query("games_played > 4 & average_attempts > 15")

QBs_Summary['WORP_est'] =  0.214*QBs_Summary['average_fantasy_points_ppr']-3.283

In [3]:
QBs_Summary_Player = ff.df_summary_playeravg(QBs_Summary)

QBs_Summary_Player['type'] = (QBs_Summary_Player['average_carries_mean'] >= 5.5).astype(int).map({1: 'rusher',0: 'non-rusher'})

worp_conditions = [(QBs_Summary_Player['WORP_est_mean'] > 1.25),
                 (QBs_Summary_Player['WORP_est_mean'] <= 1.25)&(QBs_Summary_Player['WORP_est_mean'] > 0.75),
                 (QBs_Summary_Player['WORP_est_mean'] <= 0.75)&(QBs_Summary_Player['WORP_est_mean'] > 0.5),
                 (QBs_Summary_Player['WORP_est_mean'] <= 0.5)]
QBs_Summary_Player['worp_tiers'] = np.select(worp_conditions, range(1,5))

QBs_Summary_Tier = ff.df_summary_tieravg(QBs_Summary_Player)

In [4]:
QBs_Summary_Tier_plot = QBs_Summary_Tier.query("worp_tiers > 1 & type == 'non-rusher'").reset_index()

fig = sp.make_subplots(
rows=2, cols=3,
subplot_titles=("YPA", "Pass Att.", "Comp. %", "TD Rate", "Off. Grade", "Pass. Grade"))

fig.update_layout(
    width=1500,  # Set initial width
    height=700,  # Set initial height
    template="plotly_dark",
    showlegend=False
)

# ypa
fig.add_trace(go.Scatter(
    x = QBs_Summary_Tier_plot['worp_tiers'],
    y = QBs_Summary_Tier_plot['yards_per_attempts_mean'],
    mode = 'markers',
    marker=dict(
        size=15),
    error_y=dict(
        type='data',  # Use provided data values for error bars
        array=QBs_Summary_Tier_plot['yards_per_attempts_std'],  # Symmetric error values
        visible=True,
        thickness=4,
        width=7
    ),    
    ),
    row=1, col=1
)
fig.update_xaxes(
    title_text="WORP Tiers", 
    tickvals=[2, 3, 4],
    ticktext=['A','B','C'],
    row=1, col=1)
fig.update_yaxes(title_text="Yards per Attempt", row=1, col=1)

# att
fig.add_trace(go.Scatter(
    x = QBs_Summary_Tier_plot['worp_tiers'],
    y = QBs_Summary_Tier_plot['average_attempts_mean'],
    mode = 'markers',
    marker=dict(
        size=15),
    error_y=dict(
        type='data',  # Use provided data values for error bars
        array=QBs_Summary_Tier_plot['average_attempts_std'],  # Symmetric error values
        visible=True,
        thickness=4,
        width=7
    ),    
    ),
    row=1, col=2
)
fig.update_xaxes(
    title_text="WORP Tiers", 
    tickvals=[2, 3, 4],
    ticktext=['A','B','C'],
    row=1, col=2)
fig.update_yaxes(title_text="Passing Attempts", row=1, col=2)

# comp
fig.add_trace(go.Scatter(
    x = QBs_Summary_Tier_plot['worp_tiers'],
    y = QBs_Summary_Tier_plot['completion_percentage_mean'],
    mode = 'markers',
    marker=dict(
        size=15),
    error_y=dict(
        type='data',  # Use provided data values for error bars
        array=QBs_Summary_Tier_plot['completion_percentage_std'],  # Symmetric error values
        visible=True,
        thickness=4,
        width=7
    ),    
    ),
    row=1, col=3
)
fig.update_xaxes(
    title_text="WORP Tiers", 
    tickvals=[2, 3, 4],
    ticktext=['A','B','C'],
    row=1, col=3)
fig.update_yaxes(title_text="Completion Percentage", row=1, col=3)

# td rate
fig.add_trace(go.Scatter(
    x = QBs_Summary_Tier_plot['worp_tiers'],
    y = QBs_Summary_Tier_plot['td_rate_mean'],
    mode = 'markers',
    marker=dict(
        size=15),
    error_y=dict(
        type='data',  # Use provided data values for error bars
        array=QBs_Summary_Tier_plot['td_rate_std'],  # Symmetric error values
        visible=True,
        thickness=4,
        width=7
    ),    
    ),
    row=2, col=1
)
fig.update_xaxes(
    title_text="WORP Tiers", 
    tickvals=[2, 3, 4],
    ticktext=['A','B','C'],
    row=2, col=1)
fig.update_yaxes(title_text="Touchdown Rate", row=2, col=1)

# off grade
fig.add_trace(go.Scatter(
    x = QBs_Summary_Tier_plot['worp_tiers'],
    y = QBs_Summary_Tier_plot['average_off_grade_mean'],
    mode = 'markers',
    marker=dict(
        size=15),
    error_y=dict(
        type='data',  # Use provided data values for error bars
        array=QBs_Summary_Tier_plot['average_off_grade_std'],  # Symmetric error values
        visible=True,
        thickness=4,
        width=7
    ),    
    ),
    row=2, col=2
)
fig.update_xaxes(
    title_text="WORP Tiers", 
    tickvals=[2, 3, 4],
    ticktext=['A','B','C'],
    row=2, col=2)
fig.update_yaxes(title_text="PFF Offensive Grade", row=2, col=2)

# pass grade
fig.add_trace(go.Scatter(
    x = QBs_Summary_Tier_plot['worp_tiers'],
    y = QBs_Summary_Tier_plot['average_pass_grade_mean'],
    mode = 'markers',
    marker=dict(
        size=15),
    error_y=dict(
        type='data',  # Use provided data values for error bars
        array=QBs_Summary_Tier_plot['average_pass_grade_std'],  # Symmetric error values
        visible=True,
        thickness=4,
        width=7
    ),    
    ),
    row=2, col=3
)
fig.update_xaxes(
    title_text="WORP Tiers", 
    tickvals=[2, 3, 4],
    ticktext=['A','B','C'],
    row=2, col=3)
fig.update_yaxes(title_text="PFF Passing Grade", row=2, col=3)

# pio.write_image(fig, "ClusterPred_Public_Image1.png")
# Image("ClusterPred_Public_Image1.png")